## Features Analysis

Understanding relationship between categorical features and price

In [1]:
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split

In [2]:
raw_df = pd.read_csv('../../data/raw/bmw_pricing_challenge.csv')
train_df, _ = train_test_split(raw_df, test_size=0.2, random_state=42)

categorical_features = ['fuel', 'car_type', 'paint_color']

print("ANOVA TESTS (TRAINING DATA ONLY) TO PREVENT LEAKAGE\n" + "="*50)
for cat_col in categorical_features:
    if cat_col in train_df.columns:
        # ANOVA test using only training data
        groups = [train_df[train_df[cat_col] == cat]['price'].values 
                  for cat in train_df[cat_col].unique() if pd.notna(cat)]
        f_stat, p_value = stats.f_oneway(*groups)
        print(f"{cat_col.upper()} -> F-stat: {f_stat:.2f}, p-value: {p_value:.4f}")

ANOVA TESTS (TRAINING DATA ONLY) TO PREVENT LEAKAGE
FUEL -> F-stat: 6.36, p-value: 0.0003
CAR_TYPE -> F-stat: 97.50, p-value: 0.0000
PAINT_COLOR -> F-stat: 5.04, p-value: 0.0000


In [3]:
# Analyze categorical features
categorical_features = ['fuel', 'car_type', 'paint_color']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, cat_col in enumerate(categorical_features):
    if cat_col in df.columns:
        # Group by category and calculate mean price
        price_by_cat = df.groupby(cat_col)['price'].agg(['mean', 'count']).sort_values('mean', ascending=False)
        
        # Plot
        axes[idx].barh(price_by_cat.index, price_by_cat['mean'], alpha=0.7, color='steelblue')
        axes[idx].set_xlabel('Average Price (€)')
        axes[idx].set_title(f'Avg Price by {cat_col.title()}')
        axes[idx].grid(axis='x', alpha=0.3)
        axes[idx].invert_yaxis()
        
        # Add count annotations
        for i, (cat, row) in enumerate(price_by_cat.iterrows()):
            axes[idx].text(row['mean'], i, f" n={row['count']:,.0f}", 
                          va='center', fontsize=9)

save_fig('06_categorical_features')
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Statistical significance of categorical features
print("\n" + "=" * 80)
print("CATEGORICAL FEATURE IMPACT ON PRICE")
print("=" * 80)

for cat_col in categorical_features:
    if cat_col in train_df.columns:
        print(f"\n{cat_col.upper()}:")
        print("-" * 40)
        
        # Group statistics
        group_stats = train_df.groupby(cat_col)['price'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
        print(group_stats)
        
        # ANOVA test
        groups = [train_df[train_df[cat_col] == cat]['price'].values for cat in train_df[cat_col].unique() if pd.notna(cat)]
        f_stat, p_value = stats.f_oneway(*groups)
        
        print(f"\n   ANOVA Test:")
        print(f"   F-statistic: {f_stat:.2f}")
        print(f"   p-value: {p_value:.4f}")
        
        if p_value < 0.001:
            print(f"   ✅ HIGHLY SIGNIFICANT (p < 0.001)")
            print(f"   → {cat_col} strongly affects price - important feature")
        elif p_value < 0.05:
            print(f"   ✅ SIGNIFICANT (p < 0.05)")
            print(f"   → {cat_col} affects price - include in model")
        else:
            print(f"   ⚠️  NOT SIGNIFICANT (p ≥ 0.05)")
            print(f"   → {cat_col} may not add much predictive power")


CATEGORICAL FEATURE IMPACT ON PRICE

FUEL:
----------------------------------------
           mean      std  count
fuel                           
diesel 15846.11  8969.35   4641
other  15413.86 13806.67    202

   ANOVA Test:
   F-statistic: 0.43
   p-value: 0.5143
   ⚠️  NOT SIGNIFICANT (p ≥ 0.05)
   → fuel may not add much predictive power

CAR_TYPE:
----------------------------------------
                mean      std  count
car_type                            
coupe       22172.12 12962.89    104
suv         21496.12 12888.12   1058
convertible 17136.17 12329.40     47
sedan       16017.47  8048.45   1168
van         14350.00  5230.97     44
hatchback   13289.41  6228.09    699
estate      13112.14  5376.60   1606
subcompact   9521.37  4276.23    117

   ANOVA Test:
   F-statistic: 116.48
   p-value: 0.0000
   ✅ HIGHLY SIGNIFICANT (p < 0.001)
   → car_type strongly affects price - important feature

PAINT_COLOR:
----------------------------------------
                mean     